In [ ]:
import requests
import pandas as pd
import time
import logging
from datetime import datetime, timezone
from google.cloud import bigquery
from google.colab import auth


In [ ]:
auth.authenticate_user()

# --- Configuration ---
PROJECT_ID   = "summer-sun-490113-a4"
DATASET_ID   = "african_economics_raw"
TABLE_ID     = "world_bank_indicators"
FULL_TABLE   = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"


In [ ]:
# World Bank API base
WB_BASE_URL  = "https://api.worldbank.org/v2"

# Sub-Saharan African country ISO codes (48 countries)
AFRICAN_COUNTRIES = [
    "ZA", "NG", "KE", "GH", "ET", "TZ", "UG", "SN", "CI", "CM",
    "AO", "MZ", "ZM", "ZW", "BW", "NA", "MU", "RW", "MG", "MW",
    "ML", "BF", "NE", "TD", "SD", "SS", "SO", "ER", "DJ", "SL",
    "LR", "GN", "GW", "CV", "GM", "TG", "BJ", "BI", "CD", "CG",
    "GA", "GQ", "CF", "ST", "SC", "KM", "LS", "SZ"
]

# Indicators to ingest: {indicator_code: human_readable_name}
INDICATORS = {
    "NY.GDP.MKTP.CD":   "gdp_current_usd",
    "NY.GDP.PCAP.CD":   "gdp_per_capita_usd",
    "FP.CPI.TOTL.ZG":   "inflation_rate_pct",
    "SL.UEM.TOTL.ZS":   "unemployment_rate_pct",
    "NE.TRD.GNFS.ZS":   "trade_pct_gdp",
    "BX.KLT.DINV.CD.WD":"fdi_net_inflows_usd",
    "GC.DOD.TOTL.GD.ZS":"govt_debt_pct_gdp",
    "SP.POP.TOTL":       "population_total",
}

YEAR_START   = 2000
YEAR_END     = 2023
BATCH_SIZE   = 10   # countries per API request



In [ ]:
# --- Logging setup ---
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger(__name__)



In [ ]:
from google.cloud import bigquery
def setup_bigquery(client: bigquery.Client) -> None:
    """Create dataset and table with schema if they don't exist."""

    # Create dataset
    dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
    dataset_ref.location = "US"
    dataset_ref.description = "Raw World Bank economic indicators for African countries"
    try:
        client.create_dataset(dataset_ref, exists_ok=True)
        log.info(f"Dataset ready: {DATASET_ID}")
    except Exception as e:
        log.error(f"Dataset creation failed: {e}")
        raise

    # Table schema
    schema = [
        bigquery.SchemaField("ingestion_id",        "STRING",    mode="REQUIRED", description="Unique row identifier (country_indicator_year)"),
        bigquery.SchemaField("country_iso",          "STRING",    mode="REQUIRED", description="ISO 2-letter country code"),
        bigquery.SchemaField("country_name",         "STRING",    mode="NULLABLE", description="Full country name from World Bank"),
        bigquery.SchemaField("indicator_code",       "STRING",    mode="REQUIRED", description="World Bank indicator code"),
        bigquery.SchemaField("indicator_name",       "STRING",    mode="REQUIRED", description="Human-readable indicator name"),
        bigquery.SchemaField("year",                 "INTEGER",   mode="REQUIRED", description="Data year"),
        bigquery.SchemaField("value",                "FLOAT64",   mode="NULLABLE", description="Indicator value (NULL if not reported)"),
        bigquery.SchemaField("source",               "STRING",    mode="NULLABLE", description="Data source description"),
        bigquery.SchemaField("ingested_at",          "TIMESTAMP", mode="REQUIRED", description="UTC timestamp of ingestion"),
        bigquery.SchemaField("pipeline_version",     "STRING",    mode="NULLABLE", description="Pipeline version tag"),
    ]

    table_ref = bigquery.Table(FULL_TABLE, schema=schema)
    table_ref.description = "World Bank indicators — ingested by African Economic Intelligence Pipeline"

    # Partition by year for cost-efficient querying in downstream dbt models
    table_ref.range_partitioning = bigquery.RangePartitioning(
        field="year",
        range_=bigquery.PartitionRange(start=2000, end=2030, interval=1)
    )
    table_ref.clustering_fields = ["country_iso", "indicator_name"]

    try:
        client.create_table(table_ref, exists_ok=True)
        log.info(f"Table ready: {FULL_TABLE}")
    except Exception as e:
        log.error(f"Table creation failed: {e}")
        raise


In [ ]:
def fetch_indicator(indicator_code: str, countries: list, year_start: int, year_end: int) -> list[dict]:
    """
    Fetch a single World Bank indicator for a list of countries.
    Returns a list of raw value dicts.
    Uses batch requests (semicolon-separated country codes).
    Includes retry logic with exponential back-off and 60s timeout.
    """
    records    = []
    date_range = f"{year_start}:{year_end}"
    MAX_RETRIES = 3

    for i in range(0, len(countries), BATCH_SIZE):
        batch      = countries[i : i + BATCH_SIZE]
        country_str = ";".join(batch)
        page       = 1

        while True:
            url = (
                f"{WB_BASE_URL}/country/{country_str}/indicator/{indicator_code}"
                f"?format=json&date={date_range}&per_page=500&page={page}"
            )

            # --- Retry loop ---
            resp = None
            for attempt in range(MAX_RETRIES):
                try:
                    resp = requests.get(url, timeout=60)
                    resp.raise_for_status()
                    break  # success — exit retry loop
                except requests.exceptions.RequestException as e:
                    wait = 2 ** attempt  # 1s, 2s, 4s
                    log.warning(f"Attempt {attempt+1} failed [{indicator_code} batch {i//BATCH_SIZE + 1}]: {e}. Retrying in {wait}s...")
                    time.sleep(wait)
                    if attempt == MAX_RETRIES - 1:
                        log.error(f"All {MAX_RETRIES} retries exhausted for {indicator_code}, batch {i//BATCH_SIZE + 1}. Skipping.")
                        resp = None

            if resp is None:
                break  # skip this batch, move to next

            # --- Parse response ---
            payload = resp.json()

            if not isinstance(payload, list) or len(payload) < 2 or payload[1] is None:
                log.warning(f"No data returned for {indicator_code}, batch {i//BATCH_SIZE + 1}")
                break

            meta, data = payload[0], payload[1]
            records.extend(data)

            # --- Pagination ---
            total_pages = meta.get("pages", 1)
            if page >= total_pages:
                break
            page += 1
            time.sleep(0.2)

    log.info(f"  Fetched {len(records):,} raw records for {indicator_code}")
    return records

In [ ]:
def parse_records(raw_records: list[dict], indicator_name: str, ingested_at: str) -> pd.DataFrame:
    """
    Transform raw World Bank API response into a clean DataFrame
    matching the BigQuery schema.
    """
    rows = []
    for rec in raw_records:
        country_iso  = rec.get("countryiso3code") or rec.get("country", {}).get("id", "")
        country_name = rec.get("country", {}).get("value", "")
        year_raw     = rec.get("date", "")
        value        = rec.get("value")           # None if not reported
        indicator_cd = rec.get("indicator", {}).get("id", "")
        source       = rec.get("indicator", {}).get("value", "")

        # Skip aggregate regions (World Bank includes regional aggregates)
        # ISO country codes are 2-3 chars; aggregates like "SSA" are fine, but
        # non-country entries (e.g. "1W" for World) are excluded.
        if not country_iso or len(country_iso) > 3:
            continue

        try:
            year = int(year_raw)
        except (ValueError, TypeError):
            continue

        ingestion_id = f"{country_iso}_{indicator_cd}_{year}"

        rows.append({
            "ingestion_id":    ingestion_id,
            "country_iso":     country_iso,
            "country_name":    country_name,
            "indicator_code":  indicator_cd,
            "indicator_name":  indicator_name,
            "year":            year,
            "value":           float(value) if value is not None else None,
            "source":          source,
            "ingested_at":     ingested_at,
            "pipeline_version": "v1.0.0",
        })

    df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=[
        "ingestion_id", "country_iso", "country_name", "indicator_code",
        "indicator_name", "year", "value", "source", "ingested_at", "pipeline_version"
    ])
    df["ingested_at"] = pd.to_datetime(df["ingested_at"], utc=True)
    return df


In [ ]:
def load_to_bigquery(client: bigquery.Client, df: pd.DataFrame, table_id: str) -> int:
    """
    Append DataFrame rows to BigQuery table.
    Uses WRITE_APPEND — idempotency handled via ingestion_id deduplication in dbt.
    Returns number of rows loaded.
    """
    if df.empty:
        log.warning("Empty DataFrame — nothing to load.")
        return 0

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
        schema_update_options=[bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION],
    )

    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()  # wait for completion

    rows_loaded = len(df)
    log.info(f"  Loaded {rows_loaded:,} rows → {table_id}")
    return rows_loaded


In [ ]:
def run_validation(client: bigquery.Client) -> dict:
    """
    Post-load data quality checks — returns summary dict.
    These checks are logged and can feed a monitoring dashboard.
    """
    checks = {}

    queries = {
        "total_rows":          f"SELECT COUNT(*) FROM `{FULL_TABLE}`",
        "distinct_countries":  f"SELECT COUNT(DISTINCT country_iso) FROM `{FULL_TABLE}`",
        "distinct_indicators": f"SELECT COUNT(DISTINCT indicator_code) FROM `{FULL_TABLE}`",
        "null_values_pct":     f"""
            SELECT ROUND(COUNTIF(value IS NULL) / COUNT(*) * 100, 2)
            FROM `{FULL_TABLE}`
        """,
        "year_range":          f"""
            SELECT CONCAT(CAST(MIN(year) AS STRING), '-', CAST(MAX(year) AS STRING))
            FROM `{FULL_TABLE}`
        """,
        "duplicate_ids":       f"""
            SELECT COUNT(*) FROM (
                SELECT ingestion_id, COUNT(*) c FROM `{FULL_TABLE}`
                GROUP BY ingestion_id HAVING c > 1
            )
        """,
    }

    for check_name, query in queries.items():
        result = client.query(query).result()
        value  = list(result)[0][0]
        checks[check_name] = value
        log.info(f"  QA | {check_name}: {value}")

    return checks


In [ ]:
def run_pipeline():
    """Main entry point — orchestrates full ingestion run."""
    log.info("=" * 60)
    log.info("AFRICAN ECONOMIC INTELLIGENCE PIPELINE — START")
    log.info(f"Project: {PROJECT_ID} | Table: {FULL_TABLE}")
    log.info(f"Countries: {len(AFRICAN_COUNTRIES)} | Indicators: {len(INDICATORS)}")
    log.info(f"Date range: {YEAR_START}–{YEAR_END}")
    log.info("=" * 60)

    run_start    = datetime.now(timezone.utc)
    ingested_at  = run_start.strftime("%Y-%m-%dT%H:%M:%SZ")

    # Init BigQuery client
    client = bigquery.Client(project=PROJECT_ID)

    # Setup infrastructure
    setup_bigquery(client)

    # Ingest each indicator
    total_rows = 0
    all_frames = []

    for indicator_code, indicator_name in INDICATORS.items():
        log.info(f"\nIngesting: {indicator_name} ({indicator_code})")

        raw = fetch_indicator(indicator_code, AFRICAN_COUNTRIES, YEAR_START, YEAR_END)
        df  = parse_records(raw, indicator_name, ingested_at)

        if not df.empty:
            all_frames.append(df)
            log.info(f"  Parsed {len(df):,} clean rows")
        else:
            log.warning(f"  No parseable data for {indicator_code}")

    # Bulk load all indicators in one job (more efficient than per-indicator loads)
    if all_frames:
        combined_df = pd.concat(all_frames, ignore_index=True)
        log.info(f"\nTotal rows to load: {len(combined_df):,}")
        total_rows = load_to_bigquery(client, combined_df, FULL_TABLE)

    # Post-load validation
    log.info("\nRunning data quality checks...")
    qa_results = run_validation(client)

    # Summary
    duration = (datetime.now(timezone.utc) - run_start).total_seconds()
    log.info("\n" + "=" * 60)
    log.info("PIPELINE COMPLETE")
    log.info(f"  Rows loaded:        {total_rows:,}")
    log.info(f"  Null value rate:    {qa_results.get('null_values_pct', 'N/A')}%")
    log.info(f"  Duplicate IDs:      {qa_results.get('duplicate_ids', 'N/A')}")
    log.info(f"  Duration:           {duration:.1f}s")
    log.info("=" * 60)

    return qa_results



In [ ]:
if __name__ == "__main__":
    results = run_pipeline()


In [ ]:
# from google.cloud import bigquery
# client = bigquery.Client(project=PROJECT_ID)

# preview_sql = f"""
    # SELECT
       #  country_name,
       #  country_iso,
       #  indicator_name,
       #  year,
    # ROUND(value, 2) AS value
    # FROM `{FULL_TABLE}`
    # WHERE year = 2022
       #  AND indicator_name = 'gdp_per_capita_usd'
       #  AND value IS NOT NULL
     # ORDER BY value DESC
    # LIMIT 20
 # """
# df_preview = client.query(preview_sql).to_dataframe()
# print(df_preview.to_string(index=False))

         country_name country_iso     indicator_name  year    value
           Seychelles         SYC gdp_per_capita_usd  2022 16836.67
            Mauritius         MUS gdp_per_capita_usd  2022 10246.50
                Gabon         GAB gdp_per_capita_usd  2022  8409.21
             Botswana         BWA gdp_per_capita_usd  2022  8328.71
    Equatorial Guinea         GNQ gdp_per_capita_usd  2022  7589.30
         South Africa         ZAF gdp_per_capita_usd  2022  6534.25
              Namibia         NAM gdp_per_capita_usd  2022  4349.80
           Cabo Verde         CPV gdp_per_capita_usd  2022  4323.31
             Eswatini         SWZ gdp_per_capita_usd  2022  3894.16
               Angola         AGO gdp_per_capita_usd  2022  3682.11
             Djibouti         DJI gdp_per_capita_usd  2022  3133.26
              Nigeria         NGA gdp_per_capita_usd  2022  2899.16
          Congo, Rep.         COG gdp_per_capita_usd  2022  2620.84
             Zimbabwe         ZWE gdp_per_capita